<img src="https://raw.githubusercontent.com/carubbi/MQ/b58af070c45fb17b77640c06c0016859ad366b4d/notebooks/assets/imgs/UNIFOR_logo.png" width="400">
<br>
<b>
<font size="6" face="arial" color="blue">
    Graduação em Ciência da Computação
</font>
</b>
<br>
<b>
<font size="4" face="arial">
    Disciplina: Métodos Quantitativos em Computação
</font>
</b>

**Orientador: Prof. Me. Ricardo Carubbi** <br>
*Docente da Graduação e Pós-Graduação em Ciência de Dados e Inteligência Artificial*<br>
*Laboratório de Ciência de Dados e Inteligência Artificial*<br>
*Universidade de Fortaleza*<br>

# Unidade III — Regressão linear simples e diagnóstico

Como o comprimento da nadadeira se relaciona com a massa corporal média dos pinguins do arquipélago Palmer? Vamos descrever essa relação com uma reta, quantificar a incerteza da inclinação e usar os resíduos para avaliar a adequação do modelo.

O trabalho retoma o diagrama de dispersão e a covariância da Unidade I, esperança e independência da Unidade II e erro-padrão, intervalos de confiança e testes da Unidade III.

## 1. Objetivos de aprendizagem

Ao final da aula, o aluno será capaz de:

- Avaliar no diagrama de dispersão se há relação linear entre a variável resposta e o preditor.
- Calcular e interpretar o resíduo, a diferença entre o valor observado e o valor indicado pela reta.
- Ajustar a reta e interpretar a inclinação como variação da resposta média por unidade do preditor.
- Comparar a reta com a média geral e interpretar a diferença no ajuste.
- Interpretar o intervalo de confiança da inclinação e testar, pelo valor-p, se ela é zero.
- Interpretar padrões nos resíduos para avaliar a adequação do modelo e os limites da inferência.

## 2. Agenda

### 2.1. Aula 5 — Da dispersão à reta de regressão

1. Da dispersão à pergunta de regressão — 20 min
2. Da média geral à reta de regressão — 35 min
3. Como escolher a reta por mínimos quadrados ordinários (OLS) — 45 min

### 2.2. Aula 6 — Qualidade do ajuste e inferência na regressão

1. Conferência do ajuste com `ols()» e interpretação dos coeficientes — 25 min
2. Comparação da reta com a média geral — 20 min
3. Intervalo de confiança e teste de hipótese para a inclinação — 25 min
4. Acompanhamento 1 da AP3 — 30 min

### 2.3. Abertura da Aula 9 — Diagnóstico

1. Resíduos versus valores ajustados — 8 min
2. Gráfico Q–Q — 7 min
3. Diagnóstico em dupla — 5 min

As figuras já estão disponíveis para leitura. Use a equação fornecida na seção 6 para o primeiro resíduo. Na seção 7, você obterá a mesma reta por mínimos quadrados.

## 3. Importações e configurações

In [ ]:
# Importa as bibliotecas para manipular dados e gerar gráficos.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Importa a exibição de tabelas no notebook.
from IPython.display import display

# Importa os recursos para ajustar a reta e construir o gráfico Q–Q.
from statsmodels.api import qqplot
from statsmodels.formula.api import ols

In [ ]:
# Exibe os valores das tabelas com quatro casas decimais.
pd.options.display.float_format = "{:,.4f}".format

# Define o tema e a resolução dos gráficos.
sns.set_theme(style="whitegrid", context="notebook", rc={"figure.dpi": 120})

# Mantém a mesma ordem, cor e marcador para cada espécie.
especies = ["Adelie", "Chinstrap", "Gentoo"]
cores = {"Adelie": "#B34B15", "Chinstrap": "#2469A0", "Gentoo": "#207B66"}
marcadores = {"Adelie": "o", "Chinstrap": "^", "Gentoo": "s"}

## 4. Carregamento e seleção dos dados

### 4.1. Carregamento dos dados

Carregue a [base pré-processada dos pinguins](https://raw.githubusercontent.com/carubbi/MQ/main/data/processed/penguins.csv) pela URL direta e visualize as cinco primeiras linhas. Localize as colunas de massa corporal, comprimento da nadadeira e espécie.

**Tabela 1 — Prévia da base de pinguins.** No arquivo de origem, o comprimento da nadadeira está em milímetros e a massa corporal (`BODY_MASS`), em gramas. Dados de Horst, Hill e Gorman (2022).

### 4.2. Seleção e conferência dos registros

Separe os registros com `BODY_MASS`, `FLIPPER_LENGTH` e `SPECIES` observados. Informe quantos registros foram carregados, quantos foram excluídos e quantos entrarão no ajuste. Preserve esse recorte para comparar os resultados ao longo da aula.

### 4.3. Massa corporal em quilogramas

Divida a massa corporal em gramas (`BODY_MASS`) por 1.000 e armazene o resultado na coluna `BODY_MASS_KG`. Use essa coluna nos gráficos e nos cálculos a partir da seção 5.

## 5. Da dispersão à pergunta de regressão

### 5.1. Contexto

Dois pinguins têm nadadeiras de **190 mm**. Um tem massa de **3,05 kg** e o outro, de **4,60 kg**.

A matriz reúne quatro medidas corporais. Vamos observar o painel que relaciona o comprimento da nadadeira à massa corporal. No diagrama de dispersão, cada ponto representa um pinguim. O eixo horizontal mostra o comprimento da nadadeira em milímetros, e o vertical, a massa corporal em quilogramas. As cores e os marcadores identificam as espécies.

**Pergunta de regressão.** Como a massa corporal média varia com o comprimento da nadadeira?

![Matriz de dispersão por espécie, com distribuições na diagonal.](https://raw.githubusercontent.com/carubbi/MQ/main/notebooks/assets/imgs/u3_s03/figure-1.png)

**Figura 1 — Distribuições e relações entre as medidas corporais, por espécie.** A diagonal apresenta histogramas. Os outros painéis mostram as relações entre pares de medidas. Visualização produzida com Seaborn a partir da [Figura 2 de Horst, Hill e Gorman (2022)](https://journal.r-project.org/articles/RJ-2022-020/#fig:iris-penguin-scatters).

![Dispersão da massa corporal e do comprimento da nadadeira, por espécie.](https://raw.githubusercontent.com/carubbi/MQ/main/notebooks/assets/imgs/u3_s03/figure-2.png)

**Figura 2 — Massa corporal e comprimento da nadadeira.** Cada ponto representa um pinguim. Aplicação inspirada na [Figura 4 de Horst, Hill e Gorman (2022)](https://journal.r-project.org/articles/RJ-2022-020/).

### 5.2. Fundamentação teórica

O **coeficiente de correlação de Pearson** $r$ mede a direção e a intensidade da associação linear entre duas variáveis quantitativas. Para $n\geq 2$ pares completos $(x_i,y_i)$, usamos as medidas amostrais introduzidas na Unidade I.

$$
\begin{aligned}
s_{xy} &= \frac{\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})}{n-1},\\
s_x^2 &= \frac{\sum_{i=1}^{n}(x_i-\bar{x})^2}{n-1},\qquad
s_y^2 = \frac{\sum_{i=1}^{n}(y_i-\bar{y})^2}{n-1},\\
r &= \frac{s_{xy}}{s_x s_y}.
\end{aligned}
\tag{5.1}
$$

$x_i$ e $y_i$ são os dois valores observados no mesmo par. $\bar{x}$ e $\bar{y}$ são as médias amostrais. $s_{xy}$ é a covariância amostral, $s_x^2$ e $s_y^2$ são as variâncias amostrais, e $s_x$ e $s_y$ são os desvios-padrão amostrais de $X$ e $Y$. O coeficiente é definido quando $s_x>0$ e $s_y>0$.

$r$ varia de $-1$ a $1$ e não tem unidade. O sinal indica associação linear crescente ou decrescente, e $|r|$ indica sua intensidade. Trocar $X$ e $Y$ preserva $r$. A figura a seguir mostra que uma relação em curva pode ter $r=0$.

Na regressão, a pergunta define a **variável resposta** $Y$, cuja média queremos descrever, e o **preditor** $X$, usado para estimar essa média. Uma reta é uma primeira aproximação quando a média da resposta acompanha uma direção aproximadamente linear. Curvaturas e agrupamentos no diagrama ajudam a reconhecer limitações dessa aproximação. A interpretação causal exige evidências sobre o desenho do estudo (Barbetta, Reis e Bornia, 2010, §§ 11.1–11.3).

![Pontos organizados em curva simétrica, com correlação de Pearson igual a zero.](https://raw.githubusercontent.com/carubbi/MQ/main/notebooks/assets/imgs/u3_s03/figure-3.png)

**Figura 3 — Relação em curva com correlação linear igual a zero.** Ilustração própria a partir da Figura 7.7 de Navidi (2024, § 7.1, p. 533). O que o diagrama revela sobre a relação que Pearson deixa de captar?

### 5.3. Aplicação

1. Identifique a resposta, o preditor e suas unidades.
2. Descreva a direção e a forma da relação no conjunto. Observe onde os grupos por espécie se separam e onde se sobrepõem.
3. Indique uma característica do gráfico que favorece o uso de uma reta e uma ressalva baseada nos agrupamentos.
4. Localize o pinguim com nadadeira de 181 mm e massa de 3,750 kg. Ele será retomado na seção 6.

## 6. Da média geral à reta de regressão

### 6.1. Contexto

Na seção anterior, destacamos um pinguim com nadadeira de **181 mm** e massa de **3,750 kg**. A média geral de **4,202 kg** é a mesma referência para todos os pinguins. Com uma reta, a massa estimada varia conforme o comprimento da nadadeira.

**Pergunta.** Como a massa observada desse pinguim se compara com a média geral e com a massa estimada pela reta para 181 mm?

### 6.2. Fundamentação teórica

A média amostral $\bar y$ resume as massas observadas. A **esperança** é uma média teórica. A **média condicional** $\operatorname{E}(Y\mid X=x)$ descreve a média teórica de $Y$ para um dado valor $x$ de $X$. No modelo de regressão linear simples,

$$
\operatorname{E}(Y\mid X=x)=\beta_0+\beta_1x.
\tag{5.2}
$$

O **intercepto** $\beta_0$ é a média da resposta em $x=0$. A **inclinação** $\beta_1$ indica a variação nessa média quando o preditor aumenta uma unidade.

![Esquema genérico do intercepto e da variação da média condicional quando x aumenta uma unidade.](https://raw.githubusercontent.com/carubbi/MQ/main/notebooks/assets/imgs/u3_s03/figure-4.png)

**Figura 4 — Intercepto e inclinação da média condicional.** Esquema elaborado a partir da Figura 16.2 de Morettin e Bussab (2010, § 16.1, p. 450). A obra representa o intercepto por α e a inclinação por β.

Para uma observação $i$, sejam $x_i$ o valor observado do preditor e $y_i$ a resposta observada. O **valor ajustado** é

$$
\hat y_i=\hat\beta_0+\hat\beta_1x_i,
\tag{5.3}
$$

em que $\hat\beta_0$ e $\hat\beta_1$ são as estimativas do intercepto e da inclinação calculadas na amostra. O **resíduo** é a diferença entre a resposta observada e o valor ajustado,

$$
e_i=y_i-\hat y_i.
\tag{5.4}
$$

O resíduo é medido na unidade da resposta. Ele é positivo acima da reta, negativo abaixo dela e zero sobre ela. Seu módulo é a distância vertical entre o ponto e a reta.

O modelo para a observação é

$$
Y_i=\beta_0+\beta_1X_i+\varepsilon_i.
\tag{5.5}
$$

$Y_i$ e $X_i$ representam a resposta e o preditor da observação. O **erro aleatório** $\varepsilon_i=Y_i-\operatorname{E}(Y_i\mid X_i)$ é o desvio em relação à média condicional teórica. Por essa definição, $\operatorname{E}(\varepsilon_i\mid X_i=x_i)=0$. Como essa média é desconhecida, usamos os resíduos do ajuste para estudar os erros (Barbetta, Reis e Bornia, 2010, § 11.3, p. 325–328).

### 6.3. Aplicação

Para esta atividade, use a reta ajustada já fornecida. Na equação, $x$ é o comprimento da nadadeira em mm e $\hat y$ é a massa ajustada em kg.

$$
\hat y=-5{,}780831358+0{,}049685566x.
\tag{5.6}
$$

1. Calcule a diferença entre a massa observada do pinguim de 181 mm e a média geral de 4,202 kg.
2. Substitua 181 mm na reta e calcule o resíduo. Arredonde o resultado a três casas decimais.
3. Use a figura para interpretar o sinal e a magnitude das duas diferenças.

![Pinguim de 181 mm entre a linha horizontal da média geral e a reta ajustada.](https://raw.githubusercontent.com/carubbi/MQ/main/notebooks/assets/imgs/u3_s03/figure-5.png)

**Figura 5 — Massa observada e duas referências.** Segmentos verticais ligam o pinguim à média geral e à reta fornecida. Dados de Horst, Hill e Gorman (2022).

## 7. Como escolher a reta por mínimos quadrados ordinários (OLS)

### 7.1. Contexto

Na seção anterior, usamos uma reta fornecida para calcular o resíduo de um pinguim. Agora vamos comparar duas retas usando os **342 registros**.

No gráfico, as duas retas passam pelo ponto das médias, que reúne o comprimento médio da nadadeira e a massa média. As inclinações são **0,030 e 0,060 kg/mm, ou seja, para cada 1 mm a mais de nadadeira, as retas indicam aumentos de 30 g e 60 g na massa estimada, respectivamente**. Cada reta fica mais próxima de pontos diferentes.

**Pergunta.** Como usar os resíduos de todos os pinguins para decidir qual reta se ajusta melhor ao conjunto?

![Retas candidatas de inclinações 0,030 e 0,060 kg por mm cruzam o ponto das médias.](https://raw.githubusercontent.com/carubbi/MQ/main/notebooks/assets/imgs/u3_s03/figure-6.png)

**Figura 6 — Retas candidatas.** Ambas passam pelo ponto das médias. A diferença entre elas está na inclinação. Dados de Horst, Hill e Gorman (2022).

### 7.2. Fundamentação teórica

Na seção 6.2, o erro aleatório é o desvio em relação à média condicional teórica. Para uma reta candidata com intercepto $b_0$ e inclinação $b_1$, o resíduo nos dados é $e_i(b_0,b_1)=y_i-(b_0+b_1x_i)$, a diferença entre a resposta observada e a indicada pela reta candidata.

Resíduos positivos e negativos podem se anular na soma. O método de **mínimos quadrados ordinários** escolhe a reta que minimiza a soma dos quadrados dos resíduos.

$$
\mathrm{SQR}(b_0,b_1)=\sum_{i=1}^{n}[y_i-(b_0+b_1x_i)]^2,
\tag{5.7}
$$

em que $\mathrm{SQR}(b_0,b_1)$ é a **soma dos quadrados dos resíduos** da reta candidata e $n$ é o número de observações.

O ajuste varia $b_0$ e $b_1$ para tornar essa soma a menor possível.

$$
\min \mathrm{SQR}(b_0,b_1)
=
\min \sum_{i=1}^{n}\bigl[y_i-(b_0+b_1x_i)\bigr]^2.
$$

A Figura 7 destaca as diferenças que entram nessa soma.

![Segmentos verticais entre pontos observados e uma reta candidata.](https://raw.githubusercontent.com/carubbi/MQ/main/notebooks/assets/imgs/u3_s03/figure-7.png)

**Figura 7 — Diferenças verticais entre valores observados e valores na reta.** Esquema elaborado a partir da Figura 11.3 de Montgomery e Runger (2018, § 11.2, p. 285).

Para obter a reta de mínimos quadrados, retomamos a covariância amostral $s_{xy}$ e a variância amostral do preditor $s_x^2$ da equação (5.1).

Fixamos primeiro uma inclinação candidata $b_1$. Na equação (5.7), a soma dos quadrados mede as diferenças entre o intercepto $b_0$ e os valores $y_i-b_1x_i$. Essa soma é mínima quando $b_0$ coincide com a média desses valores, ou seja, $b_0=\bar y-b_1\bar x$.

Com esse intercepto, aplicamos a definição de resíduo da equação (5.4) à reta candidata.

$$
\begin{aligned}
e_i(b_0,b_1)
&=y_i-(b_0+b_1x_i)\\
&=y_i-\bigl[(\bar y-b_1\bar x)+b_1x_i\bigr]\\
&=(y_i-\bar y)-b_1(x_i-\bar x).
\end{aligned}
$$

Ao elevar os resíduos ao quadrado e somar, obtemos

$$
\mathrm{SQR}(\bar y-b_1\bar x,b_1)
=(n-1)\left(s_x^2b_1^2-2s_{xy}b_1+s_y^2\right),
$$

em que $s_y^2$ é a variância amostral da resposta. Ao completar o quadrado, a única parcela que depende de $b_1$ fica $s_x^2\bigl(b_1-s_{xy}/s_x^2\bigr)^2$. Como $s_x^2>0$, essa parcela é sempre maior ou igual a zero. A soma dos quadrados dos resíduos é menor quando essa parcela vale zero, o que ocorre em $b_1=s_{xy}/s_x^2$.

Usando essa inclinação na expressão do intercepto, obtemos os dois coeficientes da reta de mínimos quadrados.

$$
\hat\beta_1=\frac{s_{xy}}{s_x^2},
\qquad
\hat\beta_0=\bar y-\hat\beta_1\bar x.
\tag{5.8}
$$

A reta ajustada passa pelo ponto $(\bar x,\bar y)$, em que $\bar x$ e $\bar y$ são as médias amostrais do preditor e da resposta (Barbetta, Reis e Bornia, 2010, § 11.3.1, p. 326–327).

Para qualquer reta candidata, a soma dos resíduos com sinal satisfaz

$$
\sum_{i=1}^{n}e_i(b_0,b_1)
=n\left[\bar{y}-(b_0+b_1\bar{x})\right].
\tag{5.9}
$$

Quando a reta passa pelo ponto das médias, $b_0+b_1\bar{x}=\bar{y}$, e essa soma é zero. A soma dos quadrados permite comparar retas que têm esse mesmo ponto em comum.

### 7.3. Aplicação

#### 7.3.1. Implementação sem pacote estatístico

1. Obtenha o intercepto de cada reta candidata a partir da inclinação e do ponto das médias.
2. Calcule os valores indicados pelas retas e os resíduos em cada um dos 342 registros. Some os quadrados dos resíduos de cada reta.
3. Calcule a inclinação e o intercepto da reta de mínimos quadrados.
4. Calcule os valores ajustados, os resíduos e a soma dos quadrados dos resíduos da reta obtida. Compare com as duas candidatas.

#### 7.3.2. Interpretação

1. Identifique a reta com menor soma dos quadrados dos resíduos.
2. Explique por que a soma dos resíduos com sinal é zero nas três retas.
3. Interprete a inclinação em kg/mm. Para interpretar o intercepto, verifique se zero está na faixa observada de comprimentos.
4. Calcule novamente o resíduo do pinguim de 181 mm e compare com o valor obtido na seção 6.

## 8. Ajuste e inferência nos 342 pinguins

### 8.1. Contexto

Na Aula 5, ajustamos uma reta aos **342 pinguins**, cujas nadadeiras medem de **172 a 231 mm**. A média geral de **4,202 kg** serve de referência para todos. A reta usa o comprimento da nadadeira para estimar a massa média.

A inclinação calculada depende da amostra. Se repetíssemos a coleta pelo mesmo procedimento, poderíamos obter outra inclinação.

**Perguntas.** Quanto a reta melhora o ajuste em relação à média geral? Qual é a incerteza da inclinação estimada?

![Reta ajustada aos mesmos 342 pinguins pela regressão simples.](https://raw.githubusercontent.com/carubbi/MQ/main/notebooks/assets/imgs/u3_s03/figure-8.png)

**Figura 8 — Reta ajustada aos 342 registros.** A linha é exibida no intervalo observado do preditor. Dados de Horst, Hill e Gorman (2022).

### 8.2. Fundamentação teórica

**Coeficientes.** A inclinação é a variação média estimada por unidade do preditor. O intercepto é o valor médio estimado quando o preditor vale zero. Vale conferir a faixa do preditor antes de interpretar esse valor.

**Qualidade do ajuste.** A Figura 5 mostra duas referências para a mesma observação. Retome os dois segmentos para distinguir o desvio em relação à média do resíduo em relação à reta. Essa comparação também é ilustrada na Figura 11.7 de Barbetta, Reis e Bornia (2010, p. 330). Para todos os registros, calculamos

$$
\mathrm{SQT}=\sum_{i=1}^{n}(y_i-\bar y)^2,\qquad \mathrm{SQR}=\sum_{i=1}^{n}e_i^2.
\tag{6.1}
$$

$\mathrm{SQT}$ é a soma dos quadrados em relação à média geral e $\mathrm{SQR}$ é a soma dos quadrados dos resíduos. Com $\mathrm{SQT}>0$, o **coeficiente de determinação** $R^2$ é a redução proporcional da soma de quadrados ao usar a reta no lugar da média geral. O **$R^2$ ajustado** leva em conta os graus de liberdade de cada referência. Para a regressão simples com intercepto e $n>2$,

$$
R^2=1-\frac{\mathrm{SQR}}{\mathrm{SQT}}, \qquad
R^2_{\mathrm{ajustado}}
=1-\frac{\mathrm{SQR}/(n-2)}{\mathrm{SQT}/(n-1)}.
\tag{6.2}
$$

A média geral usa um parâmetro estimado e tem $n-1$ graus de liberdade. A reta usa dois, o intercepto e a inclinação, e tem $n-2$ graus de liberdade residuais (Montgomery e Runger, 2018, § 12.2.1).

**Condições da inferência clássica.** A média condicional segue uma reta e os erros são independentes entre observações e têm variância constante, dados os valores do preditor. A normalidade condicional dos erros, junto a essas condições, sustenta a referência $t$ exata em amostras finitas. A seção 9 retoma essas condições.

**Erro-padrão da inclinação.** Nas distribuições amostrais, usamos o erro-padrão para descrever a variabilidade de uma estimativa entre amostras. Para a inclinação, estimamos

$$
\operatorname{EP}(\hat\beta_1)=
\sqrt{\frac{\mathrm{SQR}/(n-2)}{\sum_{i=1}^{n}(x_i-\bar x)^2}}.
\tag{6.3}
$$

O numerador dentro da raiz é a variância residual estimada. O denominador mede a dispersão do preditor em torno de sua média e deve ser positivo. Para uma mesma dispersão do preditor, mais variabilidade residual aumenta o erro-padrão.

**Intervalo de confiança e teste.** Retome a ideia de estimativa mais ou menos margem de erro da aula sobre a média. O intervalo para $\beta_1$ é

$$
\hat\beta_1\pm t_{1-\alpha/2,n-2}\operatorname{EP}(\hat\beta_1).
\tag{6.4}
$$

$1-\alpha$ é o nível de confiança e $t_{1-\alpha/2,n-2}$ é o quantil de ordem $1-\alpha/2$ da distribuição $t$ com $n-2$ graus de liberdade. Para testar $H_0\colon\beta_1=0$ contra $H_1\colon\beta_1\ne0$, calculamos a estatística observada

$$
t_{\mathrm{obs}}=\frac{\hat\beta_1}{\operatorname{EP}(\hat\beta_1)}.
\tag{6.5}
$$

Sob $H_0$ e as condições do modelo, $T$ segue uma distribuição $t$ com $n-2$ graus de liberdade. O **valor-p bilateral** é $\Pr(|T|\geq |t_{\mathrm{obs}}|)$, a soma das áreas nas duas caudas além de $\pm|t_{\mathrm{obs}}|$ (Barbetta, Reis e Bornia, 2010, § 11.3.3; Montgomery e Runger, 2018, §§ 11.4.1 e 11.5.1).

### 8.3. Aplicação

#### 8.3.1. Conferência do ajuste com `ols()`

1. Use `ols()` e `fit()` para ajustar a massa corporal em quilogramas (`BODY_MASS_KG`) em função do comprimento da nadadeira nos mesmos 342 registros.
2. Exiba `summary()` e localize a tabela **OLS Regression Results**.
3. Compare intercepto, inclinação, valores ajustados e resíduos com os cálculos da seção 7. A tabela exibe coeficientes arredondados. Para a conferência numérica, use os valores guardados pelo ajuste.
4. Interprete os coeficientes com suas unidades e considere a faixa observada do preditor.

#### 8.3.2. Comparação com a média geral

1. Calcule a soma dos quadrados dos desvios em relação à média geral. Recupere a soma dos quadrados dos resíduos da seção 7.
2. Calcule $R^2$ e $R^2$ ajustado. Confira os resultados em `R-squared» e `Adj. R-squared» na saída da seção 8.3.1.
3. Explique quanto a reta reduz a soma de quadrados e como a correção pelos graus de liberdade afeta o resultado.

#### 8.3.3. Teste e intervalo para a inclinação

1. Formule as hipóteses do teste bilateral para a inclinação e adote nível de significância de 5%.
2. A partir da soma dos quadrados dos resíduos e da dispersão do preditor, calcule o erro-padrão da inclinação. Obtenha a estatística $t$ e o intervalo de 95%. Use o valor crítico fornecido $t_{0{,}975,340}\approx 1{,}966966$.
3. Confira o erro-padrão em `std err`, a estatística em `t` e os limites em `[0.025 0.975]`. O resumo arredonda os valores. Confira o erro-padrão e os limites com a precisão disponível no ajuste. Localize o valor-p em `P>|t|`, associe-o às duas caudas e consulte o valor com mais precisão no objeto ajustado se a tabela exibir `0.000`.
4. Interprete o intervalo em kg/mm e a evidência contra a hipótese de inclinação zero, considerando as condições do modelo.

## 9. Dos resíduos ao diagnóstico

### 9.1. Contexto

Depois de estimar a inclinação, voltamos aos pinguins para examinar como a reta se afasta das massas observadas.

O pinguim de **181 mm** tem massa observada de **3,750 kg**. No gráfico de resíduos, sua posição horizontal é **3,212 kg**, a massa estimada pela reta. Sua posição vertical é **+0,538 kg**, a diferença entre a massa observada e a estimada. A linha em zero marca onde essas duas massas coincidem.

**Pergunta.** O que a forma e a dispersão dos resíduos dizem sobre a adequação da reta e os limites da inferência sobre a inclinação?

### 9.2. Fundamentação teórica

O erro mede o desvio em relação à média condicional teórica. Como essa média é desconhecida, o resíduo fornece evidência indireta sobre o erro. Em mínimos quadrados com intercepto, a média dos resíduos é zero por construção. O diagnóstico examina o que acontece ao longo dos valores ajustados.

**Linearidade da média.** A média condicional segue a reta da equação (5.2). No gráfico de resíduos versus valores ajustados, observe o **centro da nuvem** ao longo do eixo horizontal. Um centro próximo de zero em toda a faixa é compatível com uma reta. Uma curvatura indica que a reta tende a superestimar a resposta em algumas faixas e a subestimá-la em outras.

**Independência dos erros.** Dados os valores do preditor, a distribuição do erro de uma observação permanece a mesma quando se conhecem os erros das demais. Examine **como os dados foram coletados**. Medições repetidas de um mesmo indivíduo ou observações de um mesmo grupo podem compartilhar fatores que afetam a resposta, gerando dependência. Se faltarem informações sobre a coleta, deixe a condição em aberto.

**Normalidade dos erros.** Com variância constante, o modelo normal supõe $\varepsilon_i\mid X_i=x_i\sim\mathcal N(0,\sigma^2)$. O gráfico **Q–Q** compara os quantis dos resíduos com os quantis de mesma ordem da Normal padrão. Observe o alinhamento com a reta de referência **no centro e nas caudas**. Um alinhamento aproximado sugere compatibilidade com a Normal. Desvios sistemáticos mostram onde essa aproximação é mais fraca.

**Espalhamento constante dos erros.** A variância condicional dos erros é $\operatorname{Var}(\varepsilon_i\mid X_i=x_i)=\sigma^2$ para cada valor do preditor. No gráfico de resíduos versus valores ajustados, compare a **largura vertical da faixa** de uma região para outra. Uma largura semelhante é compatível com variância constante. Uma faixa que se alarga ou se estreita em forma de funil sugere variância desigual, o que pode comprometer os erros-padrão clássicos.

As iniciais dos quatro nomes formam o mnemônico **LINE**.

Sob uma média condicional bem especificada, independência e variância constante sustentam o cálculo clássico dos erros-padrão. A normalidade, junto às demais condições, sustenta a referência $t$ exata usada nos intervalos e testes em amostras finitas (Morettin e Bussab, 2010, § 16.5; Montgomery e Runger, 2018, § 11.7.1).

![Três padrões ilustrativos: faixa regular, funil e curvatura.](https://raw.githubusercontent.com/carubbi/MQ/main/notebooks/assets/imgs/u3_s03/figure-9.png)

**Figura 9 — Padrões ilustrativos de resíduos.** Uma faixa regular tem largura semelhante ao longo do eixo. O funil ilustra mudança na dispersão e a curvatura ilustra uma média condicional cuja forma a reta pode deixar de captar. Esquema elaborado a partir da Figura 11.9 de Montgomery e Runger (2018, p. 297).

### 9.3. Aplicação

Use os dois gráficos prontos, gerados para o mesmo ajuste. Os 342 registros analisados têm identificadores distintos. Os pinguins foram medidos em três ilhas do arquipélago Palmer ao longo de três anos, de 2007 a 2009. [Horst, Hill e Gorman (2022)](https://journal.r-project.org/articles/RJ-2022-020/)

1. **Resíduos versus valores ajustados.** Localize o pinguim retomado. Examine curvatura, mudanças na dispersão e padrões por espécie.
2. **Gráfico Q–Q.** Localize desvios da reta de referência e interprete o que sugerem sobre a normalidade dos erros.
3. **Diagnóstico.** Registre uma conclusão de cada gráfico e uma ressalva sobre a inferência. Use os identificadores, as ilhas e os anos de coleta para discutir se há indícios de medições repetidas ou agrupamentos que possam tornar as observações dependentes.

![Gráfico de resíduos versus massa ajustada com espécies destacadas.](https://raw.githubusercontent.com/carubbi/MQ/main/notebooks/assets/imgs/u3_s03/figure-10.png)

**Figura 10 — Resíduos versus valores ajustados.** O círculo identifica a observação de 181 mm. As cores e os marcadores identificam as espécies. Dados de Horst, Hill e Gorman (2022).

![Gráfico quantil–quantil Normal dos resíduos da regressão simples.](https://raw.githubusercontent.com/carubbi/MQ/main/notebooks/assets/imgs/u3_s03/figure-11.png)

**Figura 11 — Gráfico Q–Q Normal dos resíduos.** A reta de referência leva em conta a média e o desvio-padrão dos resíduos. Os eixos exibem quantis teóricos e observados. Dados de Horst, Hill e Gorman (2022).

## 10. Estudo, exercícios e referências

### 10.1. Materiais didáticos

1. **Associação e correlação.** Barbetta, Reis e Bornia, §§ 11.1–11.2.
2. **Regressão simples, ajuste e inferência.** Barbetta, Reis e Bornia, da introdução da § 11.3 até a § 11.3.3.
3. **Coeficiente de determinação ajustado.** Montgomery e Runger, § 12.2.1, trecho da p. 324 sobre o coeficiente ajustado.
4. **Diagnóstico dos resíduos.** Barbetta, Reis e Bornia, § 11.3.4, e Morettin e Bussab, § 16.5, p. 467–470.

### 10.2. Exercícios indicados

Os exercícios são do capítulo 11 de Barbetta, Reis e Bornia (2010).

1. **Interpretar uma reta.** Exercício 5(a–b), p. 344. Estime um valor e interprete a inclinação nas unidades do problema.
2. **Ajustar e testar.** Exercício 6(a, b, d), p. 344. Calcule os coeficientes e o coeficiente de determinação e teste a hipótese de inclinação zero. No item 6(d), calcule o erro-padrão da inclinação e confira-o na saída do ajuste.
3. **Diagnosticar uma reta.** Exercício 9(a–b), p. 345. Ajuste a reta e avalie sua adequação pelo gráfico de resíduos.

### 10.3. Referências

1. BARBETTA, Pedro Alberto; REIS, Marcelo Menezes; BORNIA, Antonio Cezar. *Estatística para cursos de engenharia e informática*. 3. ed. São Paulo: Atlas, 2010.
2. MORETTIN, Pedro Alberto; BUSSAB, Wilton de Oliveira. *Estatística básica*. 6. ed. São Paulo: Saraiva, 2010.
3. MONTGOMERY, Douglas C.; RUNGER, George C. *Applied Statistics and Probability for Engineers*. 7. ed. Hoboken: Wiley, 2018.
4. HORST, Allison M.; HILL, Alison Presmanes; GORMAN, Kristen B. Palmer Archipelago Penguins Data in the palmerpenguins R Package — An Alternative to Anderson's Irises. *The R Journal*, v. 14, n. 1, 2022. [Artigo](https://journal.r-project.org/articles/RJ-2022-020/).
5. NAVIDI, William. *Statistics for Engineers and Scientists*. 6. ed. Nova York: McGraw Hill, 2024.